In [ ]:
import logfire
from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool

from src.agents.agentic.grader import GraderAgent
from src.agents.agentic.planner import PlannerAgent
from src.agents.agentic.query_expander import QueryExpander
from src.agents.agentic.query_rewriter import QueryRewriter
from src.agents.agentic.router import RouterAgent
from src.agents.agentic.synthesizer import SynthesizerAgent
from src.agents.graph.graph import compile_graph_with_postgres
from src.agents.graph.runner import GraphPipeline
from src.agents.memory.short_term import ShortTermMemoryManager
from src.agents.retrieval import RetrievalAgent
from src.common.llm.gemini import GeminiClient
from src.common.llm.groq import GroqClient
from src.common.services.hybrid_search import HybridSearch
from src.common.services.qdrant import QdrantStorageService
from src.common.services.reranker import Reranker
from src.common.utils.config import config
from src.ingestion.embedding import EmbeddingService

In [ ]:
logfire.configure(service_name="Quering")

In [ ]:
pool = AsyncConnectionPool(
    conninfo=config.POSTGRES_CONN_STRING,
    min_size=2,
    max_size=10,
    open=False,
    kwargs={"autocommit": True, "row_factory": dict_row},
)

In [ ]:
gemini_client = GeminiClient(timeout_seconds=30, max_retries=2, model=config.GEMINI_MODEL)
groq_client = GroqClient(timeout_seconds=30, max_retries=2)

In [ ]:
short_term = ShortTermMemoryManager(config.REDIS_URL)

In [ ]:
embedding_service = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME,
    dimensions=config.EMBEDDING_DIMENSIONS,
    batch_size=config.EMBEDDING_BATCH_SIZE,
)

In [ ]:
storage_service = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=embedding_service.vector_size,
    collection_name=config.QDRANT_COLLECTION_NAME,
)

In [ ]:
hybrid_search = HybridSearch(storage_service=storage_service, embedding_service=embedding_service)
reranker = Reranker()
query_expander = QueryExpander(gemini_client)

In [ ]:
retrieval_agent = RetrievalAgent(
    llm_client=groq_client,
    hybrid_search=hybrid_search,
    reranker=reranker,
    query_expand=query_expander,
)

In [ ]:
await pool.open()

In [ ]:
graph = await compile_graph_with_postgres(
    pool=pool,
    short_term=short_term,
    rewriter=QueryRewriter(gemini_client),
    router=RouterAgent(groq_client),
    planner=PlannerAgent(gemini_client),
    retrieval_agent=retrieval_agent,
    grader=GraderAgent(groq_client),
    synthesizer=SynthesizerAgent(groq_client),
)

In [ ]:
graph

In [ ]:
pipeline = GraphPipeline(graph, short_term_memory=short_term)

In [ ]:
chat = await pipeline.chat(
    user_message="Explain the difference between `deepcopy` and `copy` in the context of ML model parameters?",
    session_id="",
    user_id="124536",
)

In [ ]:
chat1 = await pipeline.chat(
    user_message="Details about transformer", session_id="", user_id="124536"
)